# Xarray-Spatial I/O: GeoTIFF Performance Tuning

Three opt-in controls let you manage memory, compression speed, and large-raster writes
when working with `open_geotiff` and `to_geotiff`. Each technique works with eager NumPy
arrays and chunked Dask arrays alike.

### What you'll build

1. [dtype on read](#1.-dtype-on-read) - cast to a narrower type at load time to cut memory in half
2. [compression_level on write](#2.-compression_level) - trade write speed for file size (or vice versa)
3. [VRT tiled output](#3.-VRT-tiled-output) - stream a chunked Dask array to a tile directory without loading it all at once

![GeoTIFF performance preview](../assets/46_geotiff_performance_preview.png)

Import the I/O functions and standard libraries.

In [ ]:
import numpy as np
import xarray as xr
import os
import tempfile
import matplotlib.pyplot as plt
from xrspatial.geotiff import open_geotiff, to_geotiff

## Data

Create a synthetic 1000x1000 float64 DEM (digital elevation model) to use throughout
the notebook.

In [ ]:
# Create a 1000x1000 float64 DEM-like raster (~8MB)
rng = np.random.default_rng(42)
elevation = rng.normal(loc=500, scale=100, size=(1000, 1000)).astype(np.float64)
y = np.linspace(40.0, 41.0, 1000)
x = np.linspace(-106.0, -105.0, 1000)
dem = xr.DataArray(elevation, dims=['y', 'x'],
                   coords={'y': y, 'x': x},
                   attrs={'crs': 4326})
print(f"DEM shape: {dem.shape}, dtype: {dem.dtype}, size: {dem.nbytes / 1e6:.1f} MB")

The raster is random Gaussian noise centered at 500 m elevation. It is small enough
to fit in memory but large enough that dtype and compression choices have a visible
impact on timing and file size.

In [ ]:
dem.plot.imshow(cmap='terrain', size=5, aspect=1)
plt.title('Synthetic DEM')
plt.show()

## 1. dtype on read

Pass `dtype` to `open_geotiff` to cast the raster to a narrower type at load time.
Reading a float64 file as float32 halves memory use without any extra copy.
The cast happens inside rasterio before the array reaches Python, so it works on
all read paths: eager, dask, and GPU.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, 'dem_f64.tif')
    to_geotiff(dem, path)

    # Read at native dtype (float64)
    native = open_geotiff(path)
    print(f"Native: dtype={native.dtype}, size={native.nbytes / 1e6:.1f} MB")

    # Read as float32 -- half the memory
    downcast = open_geotiff(path, dtype='float32')
    print(f"Downcast: dtype={downcast.dtype}, size={downcast.nbytes / 1e6:.1f} MB")

    # Works with dask too
    dask_f32 = open_geotiff(path, dtype='float32', chunks=256)
    print(f"Dask chunks dtype: {dask_f32.dtype}")

## 2. compression_level

`to_geotiff` accepts a `compression_level` argument alongside `compression`.
For zstd the range is 1 to 22 (1 = fastest, 22 = smallest file).
For deflate the range is 1 to 9.
The default is the codec's own default when `compression_level` is omitted.

Use a low level when write speed matters (streaming pipelines, scratch files).
Use a high level for archival or network transfer where file size dominates.

In [ ]:
import time

with tempfile.TemporaryDirectory() as tmpdir:
    results = []
    for level in [1, 3, 10, 22]:
        path = os.path.join(tmpdir, f'dem_zstd_l{level}.tif')
        t0 = time.perf_counter()
        to_geotiff(dem, path, compression='zstd', compression_level=level)
        elapsed = time.perf_counter() - t0
        size_kb = os.path.getsize(path) / 1024
        results.append((level, elapsed, size_kb))
        print(f"  level={level:2d}  time={elapsed:.3f}s  size={size_kb:.0f} KB")

    print(f"\nLevel 1 vs 22: {results[0][2]/results[-1][2]:.1f}x size difference")

## 3. VRT tiled output

Pass a `.vrt` path to `to_geotiff` and it writes a directory of GeoTIFF tiles
plus a VRT index file that GDAL treats as a single dataset.

Each tile corresponds to one dask chunk and is written independently, so only
one chunk is in memory at a time. This makes it practical to write arrays that
are larger than RAM.

The VRT uses relative paths, so the whole output directory is portable.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    # Chunk the DEM for dask processing
    dask_dem = dem.chunk({'y': 500, 'x': 500})

    vrt_path = os.path.join(tmpdir, 'tiled_dem.vrt')
    to_geotiff(dask_dem, vrt_path, compression='zstd')

    # Show what was created
    tiles_dir = os.path.join(tmpdir, 'tiled_dem_tiles')
    print("Files created:")
    print(f"  {os.path.basename(vrt_path)}")
    for f in sorted(os.listdir(tiles_dir)):
        size = os.path.getsize(os.path.join(tiles_dir, f)) / 1024
        print(f"  tiled_dem_tiles/{f}  ({size:.0f} KB)")

    # Read it back via VRT
    result = open_geotiff(vrt_path)
    print(f"\nRound-trip: shape={result.shape}, dtype={result.dtype}")
    print(f"Max difference: {float(np.abs(result.values - dem.values).max()):.2e}")

## Summary

| Feature | Parameter | When to use |
|---|---|---|
| dtype cast | `open_geotiff(..., dtype='float32')` | Reduce read memory by half |
| compression level | `to_geotiff(..., compression_level=1)` | Fast scratch writes; set high for archival |
| VRT tiled output | `to_geotiff(..., 'out.vrt')` | Stream large dask arrays to disk without OOM |

### References

- [GDAL VRT format](https://gdal.org/en/stable/drivers/raster/vrt.html)
- [Zstandard compression](https://facebook.github.io/zstd/)
- [Cloud Optimized GeoTIFF (COG)](https://www.cogeo.org/)
- [rasterio documentation](https://rasterio.readthedocs.io/en/stable/)